In [ ]:
import os, re, json, math, time, random, sys, io
import requests
import anthropic
from IPython.display import Markdown, display, update_display
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from huggingface_hub import login
import gradio as gr
import pytest
import subprocess
import warnings
warnings.filterwarnings("ignore")


In [2]:
env_path = find_dotenv(".env", usecwd=True)
load_dotenv(env_path, override=True)

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['ANTHROPIC_API_KEY'] = os.getenv('ANTHROPIC_API_KEY', 'your-key-if-not-using-env')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN', 'your-key-if-not-using-env')

In [3]:
openai = OpenAI()
claude = anthropic.Anthropic()
OPENAI_MODEL = "gpt-4o"
CLAUDE_MODEL = "claude-sonnet-4-5-20250929"

In [4]:
message = claude.messages.create(
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Hello, Claude",
        }
    ],
    model="claude-sonnet-4-5-20250929",
)
print(message.content)

[TextBlock(citations=None, text="Hello! It's nice to meet you. How can I help you today?", type='text')]


In [5]:
system_message = '''
Eres PyTestGenius, un asistente experto en ingeniería de calidad de software (QA) especializado en Python. 
Tu objetivo principal es analizar fragmentos de código en Python y generar casos de prueba unitaria exhaustivos utilizando el framework pytest.
Tienes un conocimiento profundo de Python 3.10+ y de las mejores prácticas de testing.
Entiendes conceptos como casos de borde (edge cases), pruebas de regresión, mocking de dependencias y pruebas parametrizadas.
Eres capaz de identificar las rutas lógicas, las posibles excepciones (TypeError, ValueError, etc.) y las condiciones límite en el código proporcionado.

**SIEMPRE** genera código que sea directamente ejecutable con pytest.
**SIEMPRE** incluye importaciones necesarias.
**SIEMPRE** cubre los "casos felices" (happy paths), los casos de borde (inputs vacíos, nulos, extremos) y los casos de error (inputs inválidos que deben lanzar excepciones).
**NUNCA** inventes funcionalidades que no estén en el código proporcionado.
NUNCA** escribas explicaciones en prosa a menos que se te pida explícitamente. Tu salida principal es el código.

Estructura tu respuesta estrictamente de la siguiente manera:
1.  La primera línea debe ser `import pytest`. No necesito que indiques ```python ni nada similar. Al final tampoco debes cerrar con ``` ni nada similar.
2.  La segunda línea debe ser `from cod_evaluar import factorial` (o el nombre de la función/clase que se te proporcione).
3.  Un bloque de código Python que contenga los casos de prueba.
4.  Cada prueba debe ser una función que comience con `test_`.
5.  Usa `assert` para las validaciones y `pytest.raises` para verificar excepciones.
6.  Agrega un comentario breve encima de cada función de prueba explicando qué caso está cubriendo.
7.  La salida final deber ser el codigo Python completo para ser ejecutado como un .py, no puede comenzar ni terminar con ```.

'''

In [6]:
def user_prompt_for(cod_python):
    user_prompt = "Genera los casos de pruebas unitarias para el código python que has recibido a través de esta función."
    user_prompt += cod_python

    return user_prompt

In [7]:
def messages_for(cod_python):
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt_for(cod_python)}
    ]

In [8]:
def pruebas_unitarias_chatgpt(cod_python):    
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for(cod_python), stream=True)
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        print(fragment, end='', flush=True)
    return reply

In [9]:
def pruebas_unitarias_claude(cod_python):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=4000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_for(cod_python)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            print(text, end="", flush=True)
    return(reply)

In [10]:
python_code = ''' 
    def factorial(n: int) -> int:
    if type(n) is not int:
        raise TypeError("El input debe ser un entero.")
    if n < 0:
        raise ValueError("El input no puede ser un número negativo.")
    if n == 0:
        return 1
    
    result = 1
    for i in range(1, n + 1):
        result *= i
    return result
    '''

In [ ]:
def write_output(nombre_del_archivo: str, contenido: str):
    with open(nombre_del_archivo, "w") as archivo:
        archivo.write(contenido.strip()) # .strip() es útil para quitar espacios en blanco al inicio y al final
    

In [15]:
def generar_pruebas(cod_python, model):
    """
    Genera el código de pruebas, lo muestra en la UI en tiempo real,
    y al finalizar escribe tanto el código original como el de pruebas en archivos.
    """
    # 1. Escribe inmediatamente el código original que se va a evaluar
    write_output("cod_evaluar.py", cod_python)

    # 2. Selecciona el modelo y obtiene el stream de la IA
    if model == "GPT":
        stream = pruebas_unitarias_chatgpt(cod_python)
    elif model == "Claude":
        stream = pruebas_unitarias_claude(cod_python)
    else:
        raise ValueError("Modelo Desconocido")
    
    # 3. Acumula la respuesta mientras la cede a Gradio para la vista en tiempo real
    full_response = ""
    for chunk in stream:
        full_response += chunk
        yield full_response # Esto actualiza la caja de texto en la UI

    # 4. Una vez terminado el stream, escribe el código de pruebas completo en su archivo
    # Es crucial que el prompt de la IA le pida generar: from cod_evaluar import ...
    write_output("test_pruebas.py", full_response)

In [39]:
def ejecutar_pruebas():
    """
    Ejecuta pytest sobre el archivo 'test_pruebas.py'.
    Asume que tanto 'test_pruebas.py' como 'cod_evaluar.py' ya existen.
    """
    nombre_archivo_pruebas = "test_pruebas.py"
    nombre_modulo = "test_pruebas"

    # 1. Verifica que el archivo de pruebas exista antes de intentar ejecutarlo
    if not os.path.exists(nombre_archivo_pruebas):
        return "Error: No se ha generado el archivo de pruebas. Por favor, haz clic en 'Generar Pruebas Unitarias' primero."

    # Revisa si el módulo de pruebas está en la caché de Python.
    if nombre_modulo in sys.modules:
        # Si está, lo eliminamos por completo.
        # Esto fuerza a pytest a leer el archivo desde cero.
        del sys.modules[nombre_modulo]
        
    # 2. Redirige la salida estándar para capturarla
    original_stdout = sys.stdout
    captured_output = io.StringIO()
    sys.stdout = captured_output

    try:
        # 3. Ejecuta pytest directamente sobre el archivo de pruebas.
        # Pytest se encargará de encontrar e importar 'cod_evaluar.py'.
        pytest.main(['-v', nombre_archivo_pruebas])
    except Exception as e:
        # Captura cualquier error inesperado durante la ejecución de pytest
        return f"Ocurrió un error al ejecutar pytest: {e}"
    finally:
        # 4. Restaura siempre la salida estándar
        sys.stdout = original_stdout

    # 5. Devuelve el resultado capturado para mostrarlo en Gradio
    return captured_output.getvalue()

In [33]:
css = """
/* Target the main background of the app */
.gradio-container {
    background-color: #0B0F19 !important;
}

/* Target the background of rows and columns */
.gradio-container .form {
    background-color: #1F2937 !important;
    border-color: #374151 !important;
}

/* Target the labels of components */
.gradio-container .label-wrap {
    color: white !important;
}

/* Target the text boxes and code editors */
.gradio-container textarea, .gradio-container .cm-editor {
    background-color: #0B0F19 !important;
    color: white !important;
    border-color: #374151 !important;
}
"""

In [40]:
with gr.Blocks(css=css, title="Generador de Pruebas Unitarias") as ui:
    gr.Markdown("## 🤖 Generador de Pruebas Unitarias con IA")
    gr.Markdown("Escribe tu código en Python, selecciona un modelo de IA y genera pruebas unitarias con `pytest`.")
    
    with gr.Row():
        # Columna de la izquierda para entradas
        with gr.Column(scale=1):
            python_input = gr.Code(
                label="Tu Código Python",
                value=python_code,
                language="python",
                lines=15,
                
            )
            model_selector = gr.Dropdown(
                ["GPT", "Claude"],
                label="Selecciona el Modelo de IA",
                value="GPT"
            )
            generar_btn = gr.Button("1. Generar Pruebas Unitarias", variant="primary")

        # Columna de la derecha para salidas
        with gr.Column(scale=2):
            pruebas_output = gr.Code(
                label="Código de Pruebas Generado",
                language="python",
                lines=15,
                interactive=True,
                
            )
            ejecutar_btn = gr.Button("2. Ejecutar Pruebas", variant="secondary")
            ejecucion_output = gr.Textbox(
                label="Resultado de la Ejecución de Pytest",
                lines=10,
                placeholder="La salida de pytest aparecerá aquí...",
                interactive=False,
                
            )

    # Conexión de los eventos de los botones con las funciones
    generar_btn.click(
        fn=generar_pruebas,
        inputs=[python_input, model_selector],
        outputs=[pruebas_output]
    )
    
    # Cambia la conexión del botón de ejecución para que pase ambos códigos
    ejecutar_btn.click(
        fn=ejecutar_pruebas,
        inputs=None, # <--- No hay entradas
        outputs=[ejecucion_output]
        
    )

# Lanza la aplicación
ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


gio: http://127.0.0.1:7868/: The specified location is not supported


¡Archivo 'cod_evaluar.py' creado exitosamente! ✅
import pytest
from cod_evaluar import factorial

# Prueba del caso feliz: cálculo del factorial de un número pequeño
def test_factorial_small_number():
    assert factorial(5) == 120

# Prueba del caso feliz: cálculo del factorial de cero
def test_factorial_zero():
    assert factorial(0) == 1

# Prueba de caso límite: calculo del factorial de uno
def test_factorial_one():
    assert factorial(1) == 1

# Prueba de caso límite: cálculo del factorial de un número mayor
def test_factorial_large_number():
    assert factorial(10) == 3628800

# Prueba de error: el input es negativo, debería lanzar ValueError
def test_factorial_negative():
    with pytest.raises(ValueError, match="El input no puede ser un número negativo."):
        factorial(-5)

# Prueba de error: el input no es un entero, debería lanzar TypeError
def test_factorial_non_integer():
    with pytest.raises(TypeError, match="El input debe ser un entero."):
        factorial(3.5)